In [1]:
import pandas as pd

In [2]:
account=pd.read_csv(r"../BankData/account.csv", sep=";")
account.head()

,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,930101
1,3818,74,POPLATEK MESICNE,930101
2,704,55,POPLATEK MESICNE,930101
3,2378,16,POPLATEK MESICNE,930101
4,2632,24,POPLATEK MESICNE,930102


In [3]:
account.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   account_id   4500 non-null   int64 
 1   district_id  4500 non-null   int64 
 2   frequency    4500 non-null   object
 3   date         4500 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 140.8+ KB


In [4]:
account.duplicated().sum()

0

In [5]:
account["account_id"].nunique()

4500

In [6]:
account["district_id"] = account["district_id"].astype(str)
account["account_id"] = account["account_id"].astype(str)
account["date"] = pd.to_datetime(account["date"].astype(str).str.zfill(6), format="%y%m%d")

In [7]:
freq_map = {"POPLATEK MESICNE": "monthly","POPLATEK TYDNE": "weekly","POPLATEK PO OBRATU": "after_transaction"}
account["frequency"] = account["frequency"].map(freq_map)

In [8]:
account["frequency"].value_counts()

frequency
monthly              4167
weekly                240
after_transaction      93
Name: count, dtype: int64

In [9]:
account.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   account_id   4500 non-null   object        
 1   district_id  4500 non-null   object        
 2   frequency    4500 non-null   object        
 3   date         4500 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 140.8+ KB


In [10]:
account.sample(5)

,account_id,district_id,frequency,date
795,8564,68,monthly,1993-09-12
2023,301,60,monthly,1995-09-10
1481,2195,73,monthly,1994-10-08
532,148,8,monthly,1993-06-21
2238,3559,18,monthly,1995-12-30


## Stage 5: Data Cleaning & Transformation — `account` Table

### 1. Data Cleaning & Transformation Log

| Column Name | Raw Type | Cleaned Type | Original / Raw Value | Converted / Mapped Value | Business / Risk Rationale |
| :--- | :---: | :---: | :--- | :--- | :--- |
| **`account_id`** | `int64` | `str` | Numeric integer (e.g., `1787`) | String object (e.g., `'1787'`) | Primary key. Converted to string to prevent numerical math operations and enable clean merging with `loan`, `disp`, and `trans` tables. |
| **`district_id`** | `int64` | `str` | Numeric integer (e.g., `18`) | String object (e.g., `'18'`) | Foreign key linking to the `district` (demographics) table. |
| **`date`** | `int64` | `datetime64[ns]` | YYMMDD integer string (e.g., `930705`) | ISO standard datetime (e.g., `1993-07-05`) | Represents account opening date. Essential for calculating "Account Tenure at Application" (Character C). Zero-padded via `.str.zfill(6)`. |
| **`frequency`** | `str` | `str` | Czech statement codes:<br>• `"POPLATEK MESICNE"`<br>• `"POPLATEK TYDNE"`<br>• `"POPLATEK PO OBRATU"` | English translation mappings:<br>• `'monthly'`<br>• `'weekly'`<br>• `'after_transaction'` | Standardized Czech terms into English for business readability and executive presentation. |

---

### 2. Validation & Integrity Audit Checklist

- [x] **Null Values:** 0 missing values detected across all columns.
- [x] **Primary Key Uniqueness:** `account_id` contains 4,500 unique values (100% unique, zero duplicates).
- [x] **Date Conversion:** Successfully converted to `YYYY-MM-DD` timestamps without parsing errors.
- [x] **Categorical Translation:** `frequency` successfully mapped with 0 unmapped (`NaN`) values remaining.

In [11]:
account.to_csv("../cleanData/clean_account.csv", sep=";", index=False)